In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
project_root = Path("..").resolve()


sys.path.append(os.path.abspath(".."))

from app.services.cleaner import load_and_clean
from app.services import analytics as an

filepath = project_root / "data" / "Sorted_cart_insight_supermarket_2025_profit_tax.csv"
string_filepath  = str(filepath)
df = pd.read_csv(string_filepath)
print(an.get_sales_summary(df))
print(an.get_top_categories(df))
print(an.get_top_product_in_top_categories(df))
print(an.get_profit_margin_by_category(df))
print(an.get_discount_impact(df))
print(an.get_slow_movers(df))
print(an.get_payment_mode_breakdown(df))

{'total_gross_sales': np.int64(34450424), 'total_net_sales': np.float64(33556191.75), 'total_discount_given': np.float64(894232.25), 'total_units_sold': 225205, 'total_transactions': 85771, 'avg_order_value': np.float64(391.23), 'total_profit': np.float64(4186862.43), 'total_tax_collected': 'N/A'}
[{'category': 'Grocery', 'total_net_sales': 9486705.78, 'total_units_sold': 48593, 'total_profit': 948670.61, 'total_discount': 239317.22, 'avg_selling_price': 194.23, 'transactions': 12939, 'profit_margin_pct': 10.0, 'avg_sale_per_transaction': 733.19}, {'category': 'Staples', 'total_net_sales': 8606192.1, 'total_units_sold': 15937, 'total_profit': 688495.36, 'total_discount': 246487.9, 'avg_selling_price': 539.26, 'transactions': 4870, 'profit_margin_pct': 8.0, 'avg_sale_per_transaction': 1767.19}, {'category': 'Cleaning Supplies', 'total_net_sales': 2982578.8, 'total_units_sold': 18498, 'total_profit': 536864.2, 'total_discount': 78531.2, 'avg_selling_price': 161.22, 'transactions': 6628, 

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19828\3063448905.py:15: DtypeWarning: Columns (0: festival_season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(string_filepath)


In [ ]:
#More complex but more efficeint not in use for now 
transaction_df = (
    df.groupby("bill_number")
      .agg(
          date=("date", "first"),
          time=("time", "first"),
          day_name=("day_name", "first"),
          is_weekend=("is_weekend", "first"),
          month=("month", "first"),
          year=("year", "first"),

          festival_season=("festival_season", "first"),
          weather_tag=("weather_tag", "first"),

          store_name=("store_name", "first"),
          store_city=("store_city", "first"),
          store_state=("store_state", "first"),
          region=("region", "first"),

          gender=("gender", "first"),
          loyalty_member=("loyalty_member", "first"),
          payment_mode=("payment_mode", "first"),
          
          

          quantity_sold=("quantity_sold", "sum"),
          gross_sales=("gross_sales", "sum"),
          discount_amount=("discount_amount", "sum"),
          mrp = ("mrp",list),
          net_sales=("net_sales", "sum"),

          products=("product_name", list),
          product_ids=("product_id", list),
          categories=("category", list),
      )
      .reset_index()
)
transaction_df.head()

,bill_number,date,time,day_name,is_weekend,month,year,festival_season,weather_tag,store_name,...,loyalty_member,payment_mode,quantity_sold,gross_sales,discount_amount,mrp,net_sales,products,product_ids,categories
0,B20250000001,2025-04-13,20:48:32,Sunday,1,4,2025,NaN,Humid,Cart Insight Supermart,...,0,upi,6,426,0.0,"[52, 90]",426.0,"[Amul Butter 100g, Amul Paneer 200g]","[P025, P022]","[Dairy, Dairy]"
1,B20250000002,2025-12-15,16:37:54,Monday,0,12,2025,NaN,Pleasant,Cart Insight Supermart,...,0,upi,5,904,89.6,"[52, 560, 120]",814.4,"[Amul Butter 100g, Amul Ghee 1L, Chana Dal 1kg]","[P025, P015, P010]","[Dairy, Grocery, Grocery]"
2,B20250000003,2025-09-28,15:51:33,Sunday,1,9,2025,NaN,Cloudy,Cart Insight Supermart,...,1,cash,3,120,0.0,[40],120.0,[Kwality Walls Cornetto 110ml],[P074],[Frozen Foods]
3,B20250000004,2025-04-17,20:09:33,Thursday,0,4,2025,NaN,Hot,Cart Insight Supermart,...,0,card,8,403,27.6,"[40, 62, 35]",375.4,"[Lifebuoy Soap 125g, Mother Dairy Toned Milk 1...","[P053, P019, P078]","[Personal Care, Dairy, Fruits & Vegetables]"
4,B20250000005,2025-03-13,20:42:24,Thursday,0,3,2025,Holi,Sunny,Cart Insight Supermart,...,1,upi,2,32,0.0,[16],32.0,[Yippee Noodles 70g],[P037],[Packaged Foods]


In [3]:
import inspect

print(inspect.getsource(an.get_top_categories))

def get_top_categories(df: pd.DataFrame, top_n: int = 5) -> list:
    top = (
        df.groupby("category")
        .agg(
            total_net_sales=("net_sales", "sum"),
            total_units_sold=("quantity_sold", "sum"),
            total_profit=("profit_amount", "sum"),
            total_discount=("discount_amount", "sum"),
            avg_selling_price=("selling_price", "mean"),
            transactions=("bill_number", "nunique"),
        )
        .reset_index()
    )

    top["profit_margin_pct"] = top["total_profit"] / top["total_net_sales"] * 100

    top["avg_sale_per_transaction"] = top["total_net_sales"] / top["transactions"]

    top = top.sort_values("total_net_sales", ascending=False).head(top_n).round(2)

    return top.to_dict(orient="records")

